# AVEDAS ValveManager Demo Notebook

Dieses Notebook demonstriert die Verwendung der neuen modularen ValveManager-Klassen aus dem AVEDAS Root Cause Analysis System.

## Überblick
- **EnhancedValve**: Verbesserte Ventil-Implementierung mit Zustandshistorie und Gewichtungsberechnung
- **ValveManager**: Verwaltung mehrerer Ventile
- **Integration mit igraph**: Graph-basierte Analyse

Das Notebook zeigt die wichtigsten Features der neuen Architektur und wie sie verwendet werden können.

## 1. Imports und Setup

Importiere alle notwendigen Module, einschließlich numpy, igraph und die relevanten Klassen aus dem neuen modularen System.

In [ ]:
# Import required libraries
import numpy as np
import igraph as ig
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List

# Import the new modular components
from components.valve_manager import EnhancedValve, ValveManager
from config.constants import CarrierTypes, ValveConstants, NodeColors

# Display settings
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Alle Module erfolgreich importiert!")
print(f"📊 numpy version: {np.__version__}")
print(f"🕸️  igraph version: {ig.__version__}")
print("🔧 AVEDAS Module geladen: EnhancedValve, ValveManager")

## 2. Initialisierung von EnhancedValve und ValveManager

Erstelle Instanzen der neuen Klassen und prüfe die Grundfunktionalität.

In [ ]:
# Erstelle ein einzelnes EnhancedValve
print("🔧 Erstelle EnhancedValve...")
valve_1 = EnhancedValve(tag="XV_101", history_length=5)

print(f"✅ Ventil erstellt: {valve_1.tag}")
print(f"📈 Historie-Länge: {valve_1.history_length}")
print(f"🎯 Aktueller Zustand: {valve_1.get_current_state()}")
print(f"📝 Zustandshistorie: {valve_1.state_history}")

# Überprüfe die Carrier-Relationen
print(f"\n🔗 Verfügbare Carrier-Typen:")
for carrier, relations in valve_1.relations.items():
    print(f"  - {carrier}: {len(relations)} Relationen")

print(f"\n⚙️ Temporal Kernels:")
for carrier, kernel in valve_1.temporal_kernels.items():
    print(f"  - {carrier}: {kernel}")

# Erstelle ValveManager
print(f"\n🏭 Erstelle ValveManager...")
valve_manager = ValveManager()
print(f"✅ ValveManager erstellt mit {valve_manager.get_valve_count()} Ventilen")

## 3. Testen der Kernel-Erstellung

Teste die Methode _create_history_kernel mit unterschiedlichen Parametern und visualisiere die erzeugten Kernel.

In [ ]:
# Teste verschiedene Kernel-Konfigurationen
print("🧮 Teste verschiedene Kernel-Konfigurationen...")

# Verschiedene Kernel erstellen
test_valve = EnhancedValve("test_valve", history_length=10)

kernel_configs = [
    {"name": "Linear ansteigend", "start": 0.1, "stop": 1.0, "ramp_start": 2, "ramp_length": 6},
    {"name": "Exponentiell", "start": 0.01, "stop": 0.99, "ramp_start": 1, "ramp_length": 8},
    {"name": "Konstant", "start": 0.5, "stop": 0.5, "ramp_start": 0, "ramp_length": 10},
    {"name": "Step-Function", "start": 0.1, "stop": 0.9, "ramp_start": 5, "ramp_length": 1}
]

# Erstelle und visualisiere Kernels
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, config in enumerate(kernel_configs):
    kernel = test_valve._create_history_kernel(
        config["start"], config["stop"], 
        config["ramp_start"], config["ramp_length"]
    )
    
    axes[i].plot(range(len(kernel)), kernel, 'bo-', linewidth=2, markersize=6)
    axes[i].set_title(f'{config["name"]}')
    axes[i].set_xlabel('Historie-Index')
    axes[i].set_ylabel('Kernel-Wert')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(0, 1.1)
    
    print(f"✅ {config['name']}: {kernel}")

plt.tight_layout()
plt.suptitle('Verschiedene Kernel-Konfigurationen', y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Teste Standard-Kernels des Ventils
print(f"\n📊 Standard Temporal Kernels des Ventils:")
for carrier, kernel in valve_1.temporal_kernels.items():
    print(f"  {carrier}: {kernel}")

print(f"\n📊 Standard Value Kernels des Ventils:")
for carrier, kernel in valve_1.value_kernels.items():
    print(f"  {carrier}: {kernel}")

## 4. Testen der Zustandsaktualisierung und -historie

Führe update_state mehrfach aus und prüfe state_history sowie get_current_state.

In [ ]:
# Simuliere Zustandsänderungen über Zeit
print("⏰ Simuliere Zustandsänderungen über Zeit...")

# Anfangszustand
print(f"🎯 Anfangszustand: {valve_1.get_current_state()}")
print(f"📝 Historie: {valve_1.state_history}")

# Simuliere verschiedene Zustände
test_states = [0.2, 0.5, 0.8, 0.3, 0.9, 0.1, 0.7]
state_history_log = []

for i, new_state in enumerate(test_states):
    valve_1.update_state(new_state)
    current_history = valve_1.state_history.copy()
    state_history_log.append(current_history)
    
    print(f"📊 Schritt {i+1}: Neuer Zustand = {new_state:.1f}")
    print(f"   👁️  Aktueller Zustand: {valve_1.get_current_state()}")
    print(f"   📝 Historie: {current_history}")
    print()

# Visualisiere die Entwicklung der Zustandshistorie
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Zustandsverlauf über Zeit
all_states = [valve_1.state_history[0]] + test_states
ax1.plot(range(len(all_states)), all_states, 'ro-', linewidth=2, markersize=8)
ax1.set_title('Ventilzustand über Zeit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Zeitschritt')
ax1.set_ylabel('Ventilzustand')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Plot 2: Entwicklung der Historie (Heatmap-style)
history_matrix = np.array([valve_1.state_history] + state_history_log)
im = ax2.imshow(history_matrix, aspect='auto', cmap='viridis', interpolation='nearest')
ax2.set_title('Entwicklung der Zustandshistorie', fontsize=14, fontweight='bold')
ax2.set_xlabel('Historie-Position')
ax2.set_ylabel('Update-Schritt')
ax2.set_yticks(range(len(history_matrix)))
ax2.set_yticklabels(['Initial'] + [f'Schritt {i+1}' for i in range(len(test_states))])

# Colorbar hinzufügen
plt.colorbar(im, ax=ax2, label='Ventilzustand')

plt.tight_layout()
plt.show()

# Teste Reset-Funktionalität
print("🔄 Teste Reset-Funktionalität...")
print(f"Vor Reset: {valve_1.state_history}")
valve_1.reset_state_history(0.5)
print(f"Nach Reset (auf 0.5): {valve_1.state_history}")

## 5. Simulation eines Beispiel-Graphen und Relationserkennung

Erzeuge einen Beispiel-igraph-Graphen mit passenden Attributen und simuliere find_related_edges.

In [ ]:
# Erstelle einen Beispiel-Graph für Tests
print("🕸️ Erstelle Beispiel-Graph...")

# Graph erstellen
test_graph = ig.Graph(directed=True)

# Vertices hinzufügen mit passenden Attributen
vertices = [
    {"Tag": "XV_101", "Carrier": CarrierTypes.STATE, "AS": 0, "sensorlabel": 0, "parameterlabel": 0},
    {"Tag": "Sensor_T1", "Carrier": CarrierTypes.TEMPERATURE, "AS": 1, "sensorlabel": 1, "parameterlabel": 0}, 
    {"Tag": "Sensor_P1", "Carrier": CarrierTypes.PRESSURE, "AS": 0, "sensorlabel": 1, "parameterlabel": 0},
    {"Tag": "Param_Flow", "Carrier": CarrierTypes.FLOW, "AS": 0, "sensorlabel": 0, "parameterlabel": 1},
    {"Tag": "Sensor_L1", "Carrier": CarrierTypes.LEVEL, "AS": 2, "sensorlabel": 1, "parameterlabel": 0}
]

# Vertices hinzufügen
for vertex in vertices:
    test_graph.add_vertex(**vertex)

# Kanten hinzufügen mit carrier-Attributen
edges_data = [
    (0, 1, CarrierTypes.TEMPERATURE),  # XV_101 -> Sensor_T1
    (0, 2, CarrierTypes.PRESSURE),     # XV_101 -> Sensor_P1  
    (1, 3, CarrierTypes.FLOW),         # Sensor_T1 -> Param_Flow
    (2, 4, CarrierTypes.LEVEL),        # Sensor_P1 -> Sensor_L1
    (1, 4, CarrierTypes.TEMPERATURE)   # Sensor_T1 -> Sensor_L1
]

for source, target, carrier in edges_data:
    test_graph.add_edge(source, target, carrier=carrier, weight=1.0, Actors="")

print(f"✅ Graph erstellt mit {test_graph.vcount()} Vertices und {test_graph.ecount()} Edges")

# Graph-Details anzeigen
print(f"\n📊 Graph-Details:")
print(f"Vertices:")
for i, v in enumerate(test_graph.vs):
    print(f"  {i}: {v['Tag']} (Carrier: {v['Carrier']}, AS: {v['AS']})")

print(f"\nEdges:")
for i, e in enumerate(test_graph.es):
    source_tag = test_graph.vs[e.source]['Tag']
    target_tag = test_graph.vs[e.target]['Tag']
    print(f"  {i}: {source_tag} -> {target_tag} (Carrier: {e['carrier']})")

# Teste find_related_edges für unser Ventil
print(f"\n🔍 Teste Relationserkennung für Ventil XV_101...")
test_valve = EnhancedValve("XV_101", history_length=3)

# Vor der Relationserkennung
print(f"Vor find_related_edges:")
for carrier, relations in test_valve.relations.items():
    print(f"  {carrier}: {relations}")

# Relationserkennung durchführen
test_valve.find_related_edges(test_graph)

# Nach der Relationserkennung
print(f"\nNach find_related_edges:")
for carrier, relations in test_valve.relations.items():
    print(f"  {carrier}: {relations} (Anzahl: {len(relations)})")
    if relations:
        for edge_idx in relations:
            edge = test_graph.es[edge_idx]
            source_tag = test_graph.vs[edge.source]['Tag']
            target_tag = test_graph.vs[edge.target]['Tag']
            print(f"    Edge {edge_idx}: {source_tag} -> {target_tag}")

## 6. Anwenden von Gewichten auf Graph-Kanten

Nutze apply_weights_to_graph und prüfe, ob die Gewichte und Actor-Attribute korrekt gesetzt werden.

In [ ]:
# Teste Gewichtungsberechnung und -anwendung
print("⚖️ Teste Gewichtungsberechnung und -anwendung...")

# Setze verschiedene Zustandshistorien für Tests
test_histories = [
    [0.1, 0.2, 0.3],  # Ansteigend
    [0.8, 0.5, 0.2],  # Abfallend  
    [0.5, 0.5, 0.5],  # Konstant
    [0.0, 1.0, 0.0]   # Oszillierend
]

print("📊 Gewichte vor der Anwendung:")
for i, edge in enumerate(test_graph.es):
    print(f"  Edge {i}: weight={edge['weight']:.2f}, Actors='{edge['Actors']}'")

# Teste verschiedene Zustandshistorien
results = []
for i, history in enumerate(test_histories):
    print(f"\n🧪 Test {i+1}: Historie {history}")
    
    # Setze neue Historie
    test_valve.state_history = history.copy()
    
    # Berechne Gewichte für jeden Carrier
    weights = {}
    for carrier in test_valve.relations.keys():
        weight = test_valve._calculate_edge_weight(carrier)
        weights[carrier] = weight
        print(f"  {carrier}: {weight:.4f}")
    
    results.append({"history": history, "weights": weights})
    
    # Wende Gewichte auf Graph an
    test_valve.apply_weights_to_graph(test_graph)
    
    print(f"  📊 Gewichte nach Anwendung:")
    for j, edge in enumerate(test_graph.es):
        if j in [rel for rels in test_valve.relations.values() for rel in rels]:
            print(f"    Edge {j}: weight={edge['weight']:.4f}, Actors='{edge['Actors']}'")

# Visualisiere die Gewichtsentwicklung
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

carriers = list(CarrierTypes.TEMPERATURE, CarrierTypes.PRESSURE, CarrierTypes.FLOW, CarrierTypes.LEVEL)[:4]

for i, carrier in enumerate(carriers):
    histories = [r["history"] for r in results]
    weights = [r["weights"][carrier] for r in results]
    
    # Plot für jeden Carrier
    for j, (history, weight) in enumerate(zip(histories, weights)):
        axes[i].bar(j, weight, alpha=0.7, label=f'Test {j+1}: {history}')
    
    axes[i].set_title(f'Gewichte für {carrier}')
    axes[i].set_xlabel('Test-Fall')
    axes[i].set_ylabel('Berechnetes Gewicht')
    axes[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Detaillierte Analyse eines Gewichts
print(f"\n🔬 Detaillierte Analyse der Gewichtberechnung:")
test_valve.state_history = [0.2, 0.6, 0.9]  # Beispiel-Historie
carrier = CarrierTypes.TEMPERATURE

print(f"Historie: {test_valve.state_history}")
print(f"Temporal Kernel ({carrier}): {test_valve.temporal_kernels[carrier]}")
print(f"Value Kernel ({carrier}): {test_valve.value_kernels[carrier]}")

# Schritt-für-Schritt Berechnung nachvollziehen
value_history = []
for hist_val in test_valve.state_history:
    scaled_val = int(np.round(hist_val * test_valve.history_length))
    
    if scaled_val <= 0:
        hist_index = 0
    elif scaled_val <= 0.5:
        max_idx = len(test_valve.value_kernels[carrier]) - 1
        hist_index = int(np.round(scaled_val / 0.5 * max_idx))
    else:
        hist_index = len(test_valve.value_kernels[carrier]) - 1
    
    value_history.append(test_valve.value_kernels[carrier][hist_index])
    print(f"  {hist_val:.1f} -> scaled: {scaled_val} -> index: {hist_index} -> value: {test_valve.value_kernels[carrier][hist_index]}")

numerator = np.sum(np.array(value_history) * np.array(test_valve.temporal_kernels[carrier]))
denominator = np.sum(test_valve.temporal_kernels[carrier]) or 1.0
final_weight = round(numerator / denominator, 2)

print(f"Value History: {value_history}")
print(f"Numerator: {numerator:.4f}")
print(f"Denominator: {denominator:.4f}")
print(f"Final Weight: {final_weight:.4f}")

## 7. Mehrere Ventile verwalten und aktualisieren

Erstelle mehrere Ventile im ValveManager, aktualisiere deren Zustände und prüfe die Verwaltung und Massenaktualisierung.

In [ ]:
# Erstelle und verwalte mehrere Ventile
print("🏭 Teste ValveManager mit mehreren Ventilen...")

# Erstelle einen neuen ValveManager
multi_valve_manager = ValveManager()

# Definiere verschiedene Ventile
valve_configs = [
    {"tag": "XV_101", "history_length": 3},
    {"tag": "XV_102", "history_length": 4}, 
    {"tag": "XV_103", "history_length": 5},
    {"tag": "XV_201", "history_length": 3},
    {"tag": "XV_202", "history_length": 6}
]

# Erstelle Ventile
print("➕ Erstelle Ventile:")
for config in valve_configs:
    valve = multi_valve_manager.create_valve(config["tag"], config["history_length"])
    print(f"  ✅ {valve.tag} (Historie: {valve.history_length})")

print(f"\n📊 ValveManager Status:")
print(f"  Anzahl Ventile: {multi_valve_manager.get_valve_count()}")
print(f"  Ventil-Tags: {multi_valve_manager.get_valve_tags()}")

# Teste Einzelzugriff
print(f"\n🎯 Teste Einzelzugriff:")
try:
    valve_101 = multi_valve_manager.get_valve("XV_101")
    print(f"  ✅ Ventil XV_101 gefunden: {valve_101.tag}")
    print(f"     Aktueller Zustand: {valve_101.get_current_state()}")
except KeyError as e:
    print(f"  ❌ Fehler: {e}")

# Teste Massenaktualisierung
print(f"\n🔄 Teste Massenaktualisierung:")
new_states = {
    "XV_101": 0.25,
    "XV_102": 0.50, 
    "XV_103": 0.75,
    "XV_201": 0.90,
    "XV_202": 0.10,
    "XV_999": 0.60  # Dieses Ventil existiert nicht
}

print("Vor der Aktualisierung:")
for tag in multi_valve_manager.get_valve_tags():
    valve = multi_valve_manager.get_valve(tag)
    print(f"  {tag}: {valve.get_current_state()}")

multi_valve_manager.update_valve_states(new_states)

print("Nach der Aktualisierung:")
for tag in multi_valve_manager.get_valve_tags():
    valve = multi_valve_manager.get_valve(tag)
    print(f"  {tag}: {valve.get_current_state()}")

# Visualisiere Ventilzustände
print(f"\n📈 Visualisiere Ventilzustände:")

# Erstelle eine Simulation über mehrere Zeitschritte
time_steps = 10
valve_data = {tag: [] for tag in multi_valve_manager.get_valve_tags()}

# Simuliere zufällige Zustandsänderungen
np.random.seed(42)  # Für reproduzierbare Ergebnisse
for step in range(time_steps):
    updates = {}
    for tag in multi_valve_manager.get_valve_tags():
        # Zufällige Änderung um aktuellen Wert
        current = multi_valve_manager.get_valve(tag).get_current_state()
        change = np.random.normal(0, 0.1)  # Kleine zufällige Änderung
        new_value = np.clip(current + change, 0, 1)  # Zwischen 0 und 1 halten
        updates[tag] = new_value
        valve_data[tag].append(new_value)
    
    multi_valve_manager.update_valve_states(updates)

# Plotte die Entwicklung
plt.figure(figsize=(12, 6))
for tag, values in valve_data.items():
    plt.plot(range(time_steps), values, 'o-', label=tag, linewidth=2, markersize=6)

plt.title('Ventilzustände über Zeit', fontsize=14, fontweight='bold')
plt.xlabel('Zeitschritt')
plt.ylabel('Ventilzustand')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.show()

# Teste Historie-Anzeige
print(f"\n📝 Aktuelle Zustandshistorien:")
for tag in multi_valve_manager.get_valve_tags():
    valve = multi_valve_manager.get_valve(tag)
    print(f"  {tag}: {valve.state_history}")

## 8. Reset und Initialisierung der Ventilzustände

Teste reset_state_history und initialize_valves_from_graph, um die Initialisierung und das Zurücksetzen zu demonstrieren.

In [ ]:
# Teste Reset-Funktionen und Graph-basierte Initialisierung
print("🔄 Teste Reset-Funktionen und Graph-basierte Initialisierung...")

# 1. Teste Reset für einzelne Ventile
print("\n1️⃣ Reset einzelner Ventile:")
test_valve_reset = multi_valve_manager.get_valve("XV_101")
print(f"Vor Reset: {test_valve_reset.state_history}")

test_valve_reset.reset_state_history(0.8)
print(f"Nach Reset (0.8): {test_valve_reset.state_history}")

test_valve_reset.reset_state_history()  # Standard-Wert
print(f"Nach Standard-Reset: {test_valve_reset.state_history}")

# 2. Erstelle erweiterten Graph mit State-Knoten
print(f"\n2️⃣ Erstelle erweiterten Graph mit State-Knoten:")
extended_graph = ig.Graph(directed=True)

# Vertices mit State-Knoten hinzufügen
extended_vertices = [
    {"Tag": "XV_301_State", "Carrier": CarrierTypes.STATE, "AS": 0, "sensorlabel": 0, "parameterlabel": 0},
    {"Tag": "XV_302_State", "Carrier": CarrierTypes.STATE, "AS": 0, "sensorlabel": 0, "parameterlabel": 0}, 
    {"Tag": "XV_303_State", "Carrier": CarrierTypes.STATE, "AS": 0, "sensorlabel": 0, "parameterlabel": 0},
    {"Tag": "Sensor_T301", "Carrier": CarrierTypes.TEMPERATURE, "AS": 1, "sensorlabel": 1, "parameterlabel": 0},
    {"Tag": "Param_P301", "Carrier": CarrierTypes.PRESSURE, "AS": 0, "sensorlabel": 0, "parameterlabel": 1}
]

for vertex in extended_vertices:
    extended_graph.add_vertex(**vertex)

print(f"Graph mit {extended_graph.vcount()} Vertices erstellt:")
for i, v in enumerate(extended_graph.vs):
    print(f"  {i}: {v['Tag']} (Carrier: {v['Carrier']})")

# 3. Teste automatische Ventil-Initialisierung aus Graph
print(f"\n3️⃣ Teste automatische Ventil-Initialisierung aus Graph:")
auto_valve_manager = ValveManager()

print(f"Vor Initialisierung: {auto_valve_manager.get_valve_count()} Ventile")
print(f"Ventil-Tags: {auto_valve_manager.get_valve_tags()}")

auto_valve_manager.initialize_valves_from_graph(extended_graph)

print(f"Nach Initialisierung: {auto_valve_manager.get_valve_count()} Ventile")
print(f"Ventil-Tags: {auto_valve_manager.get_valve_tags()}")

# Zeige die automatisch erstellten Ventile
for tag in auto_valve_manager.get_valve_tags():
    valve = auto_valve_manager.get_valve(tag)
    print(f"  {tag}: Historie={valve.state_history}, Aktuell={valve.get_current_state()}")

# 4. Teste find_all_relations und apply_all_weights
print(f"\n4️⃣ Teste Massen-Operationen auf Graph:")

# Erweitere Graph mit Kanten
extended_graph.add_edge(0, 3, carrier=CarrierTypes.TEMPERATURE, weight=1.0, Actors="")  
extended_graph.add_edge(1, 4, carrier=CarrierTypes.PRESSURE, weight=1.0, Actors="")
extended_graph.add_edge(2, 3, carrier=CarrierTypes.TEMPERATURE, weight=1.0, Actors="")

print("Graph-Kanten:")
for i, e in enumerate(extended_graph.es):
    source_tag = extended_graph.vs[e.source]['Tag']
    target_tag = extended_graph.vs[e.target]['Tag']
    print(f"  Edge {i}: {source_tag} -> {target_tag} (Carrier: {e['carrier']}, Weight: {e['weight']})")

# Finde alle Relationen
auto_valve_manager.find_all_relations(extended_graph)

print(f"\nGefundene Relationen:")
for tag in auto_valve_manager.get_valve_tags():
    valve = auto_valve_manager.get_valve(tag)
    print(f"  {tag}:")
    for carrier, relations in valve.relations.items():
        if relations:
            print(f"    {carrier}: Edges {relations}")

# Setze unterschiedliche Zustände und wende Gewichte an
state_updates = {
    "XV_301": 0.3,
    "XV_302": 0.7, 
    "XV_303": 0.9
}

print(f"\n🎯 Aktualisiere Zustände und wende Gewichte an:")
print("Gewichte vor Anwendung:")
for i, e in enumerate(extended_graph.es):
    print(f"  Edge {i}: weight={e['weight']:.2f}, Actors='{e['Actors']}'")

auto_valve_manager.update_valve_states(state_updates)
auto_valve_manager.apply_all_weights(extended_graph)

print(f"Gewichte nach Anwendung:")
for i, e in enumerate(extended_graph.es):
    print(f"  Edge {i}: weight={e['weight']:.2f}, Actors='{e['Actors']}'")

# 5. Teste Clear-Funktion
print(f"\n5️⃣ Teste Clear-Funktion:")
print(f"Vor Clear: {auto_valve_manager.get_valve_count()} Ventile")
auto_valve_manager.clear_all_valves()
print(f"Nach Clear: {auto_valve_manager.get_valve_count()} Ventile")

# Zusammenfassung
print(f"\n" + "="*60)
print("🎉 ZUSAMMENFASSUNG DER TESTS")
print("="*60)
print("✅ EnhancedValve-Initialisierung erfolgreich")
print("✅ Kernel-Erstellung und -Visualisierung funktional") 
print("✅ Zustandshistorie und -updates funktional")
print("✅ Graph-Relationserkennung funktional")
print("✅ Gewichtungsberechnung und -anwendung funktional")
print("✅ ValveManager mit mehreren Ventilen funktional")
print("✅ Massen-Updates und -Operationen funktional")
print("✅ Reset- und Clear-Funktionen funktional")
print("✅ Automatische Graph-basierte Initialisierung funktional")
print(f"\n🏆 Alle Tests erfolgreich abgeschlossen!")
print("Das neue modulare ValveManager-System ist einsatzbereit.")